# backward-fn-signature — faded example 3: Complete divide_back1 (gradient wrt the denominator)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-fn-signature`. Running the beacon reports progress on the `Backprop: backward fn signature` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backward fn signature` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-fn-signature`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-fn-signature"
DD_SUBTOPIC = "Backprop: backward fn signature"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For the binary op `out = x / y` the partial wrt the denominator is `d(out)/dy = -x/y**2`. So raw `grad_y = grad_out * (-x/y**2)`. When `x` and `y` broadcast, the result must be reduced back to `y.shape` with `_unbroadcast`. The fn uses the extended binary signature `(grad_out, out, x, y)`.

## Faded exercise 3

Implement `divide_back1(grad_out, out, x, y)`, the backward fn returning the gradient of `out = x / y` with respect to `y`. The `_unbroadcast` helper and the final reduction are already provided; complete the raw (pre-reduction) gradient computation. The returned tensor must have shape `y.shape`.

**Fill in:** The raw pre-reduction gradient wrt y: grad_out times the local derivative -x/y**2.

In [ ]:
def _unbroadcast(grad, target_shape):
    while grad.ndim > len(target_shape):
        grad = grad.sum(dim=0)
    for i, s in enumerate(target_shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def divide_back1(grad_out, out, x, y):
    # out = x / y; d(out)/dy = -x / y**2.
    raw = None  # TODO: raw grad wrt y = grad_out * (-x / y**2)
    return _unbroadcast(raw, y.shape)

t.manual_seed(0)
x = t.randn(4, 3)
y = t.rand(3) + 0.5
out = x / y
grad_out = t.randn(4, 3)
print(divide_back1(grad_out, out, x, y))


def _test():
    t.manual_seed(3)
    x = t.randn(4, 3, requires_grad=True)
    y = (t.rand(3) + 0.5).requires_grad_(True)
    out = x / y
    grad_out = t.randn(4, 3)
    out.backward(grad_out)
    ref = y.grad
    got = divide_back1(grad_out, out.detach(), x.detach(), y.detach())
    assert got.shape == y.shape, (got.shape, y.shape)
    assert t.allclose(got, ref, atol=1e-5), (got, ref)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def _unbroadcast(grad, target_shape):
    while grad.ndim > len(target_shape):
        grad = grad.sum(dim=0)
    for i, s in enumerate(target_shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def divide_back1(grad_out, out, x, y):
    # out = x / y; d(out)/dy = -x / y**2.
    raw = grad_out * (-x / y ** 2)
    return _unbroadcast(raw, y.shape)

t.manual_seed(0)
x = t.randn(4, 3)
y = t.rand(3) + 0.5
out = x / y
grad_out = t.randn(4, 3)
print(divide_back1(grad_out, out, x, y))
```
</details>